# SimCLR Baseline — STL-10 Screening
**Stage I:** 100 epochs, ResNet-18, batch 256, seed 42

Why 100 epochs (not 200): STL-10 has 100k unlabeled images. At batch 256,
that is ~390 iterations/epoch × 100 epochs = ~39k iterations — roughly the
same wall-clock time as 200 epochs of CIFAR-10. Signal is sufficient for
screening; trend comparisons to CIFAR-10 remain valid.

Expected runtime: ~4–6h on T4. Fits in one 12h session.

> **Note:** STL-10 dataset is ~2.5 GB and downloads automatically on first run.

In [ ]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
import os, subprocess, glob, shutil

REPO_URL = "https://github.com/SAlaMusa/IDL.git"
REPO_DIR = "/kaggle/working/IDL"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
from models.resnet_simclr import ResNetSimCLR
from optimizers.lars import LARS
print("Imports OK")

## STL-10 Pretraining — 100 epochs
lr = 0.3 × 256/256 = 0.3 (auto-computed), batch 256, no CIFAR stem (96×96 images use standard 7×7 conv)

In [ ]:
result = subprocess.run([
    "python", "run.py",
    "--config", "configs/baseline_stl10.yaml",
    "--epochs", "100", "--batch-size", "256", "-j", "2", "--seed", "42",
], capture_output=False)

if result.returncode != 0:
    raise RuntimeError("STL-10 pretraining failed.")

checkpoints = glob.glob("runs/**/*.pth.tar", recursive=True)
latest_ckpt = max(checkpoints, key=os.path.getmtime)
dst = "/kaggle/working/stl10_resnet18_ep100_seed42.pth.tar"
shutil.copy2(latest_ckpt, dst)
print(f"Checkpoint saved: {dst}")

## STL-10 Linear Evaluation
Uses labeled train split (5,000 images) and test split. Standard normalization only.

In [ ]:
result = subprocess.run([
    "python", "linear_eval.py",
    "--checkpoint", "/kaggle/working/stl10_resnet18_ep100_seed42.pth.tar",
    "--dataset", "stl10", "--arch", "resnet18",
    "--epochs", "100", "-b", "256", "-j", "2", "--seed", "42",
], capture_output=False)

if result.returncode != 0:
    raise RuntimeError("STL-10 linear eval failed.")

if os.path.exists("linear_eval_results.csv"):
    shutil.copy2("linear_eval_results.csv", "/kaggle/working/linear_eval_results_stl10.csv")
    print("Results CSV saved.")

In [ ]:
import csv
with open("/kaggle/working/linear_eval_results_stl10.csv") as f:
    for row in csv.DictReader(f):
        print(f"Dataset: {row['dataset']}  |  Best Top-1: {row['best_top1']}%")